In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copy import deepcopy
from rashomon import hasse, extract_pools, loss, aggregate, AIS, MCMC
M = 3
R = np.array([4, 3, 3])

num_profiles = 2**M
profiles, profile_map = hasse.enumerate_profiles(M)

all_policies = hasse.enumerate_policies(M, R)
num_policies = len(all_policies)

# Profile (0, 0, 0)
sigma_000 = None
mu_000 = np.array([0])
var_000 = np.array([1])

# Profile (0, 0, 1)
sigma_001 = np.array([[1]])
mu_001 = np.array([-2])
var_001 = np.array([1])

# Profile (0, 1, 0)
sigma_010 = np.array([[1]])
mu_010 = np.array([-3])
var_010 = np.array([1])

# Profile (0, 1, 1)
sigma_011 = np.array([[1], [0]])
mu_011 = np.array([-1, 2])
var_011 = np.array([1, 1])

# Profile (1, 0, 0)
sigma_100 = np.array([[0, 1]])
mu_100 = np.array([3, 4])
var_100 = np.array([1, 1])

# Profile (1, 0, 1)
sigma_101 = np.array([[0, 1], [0, np.inf]])
mu_101 = np.array([-5, 2.5, 1.5, -2.5])
var_101 = np.array([1, 1, 1, 1])

# Profile (1, 1, 0)
sigma_110 = np.array([[0, 1], [1, np.inf]])
mu_110 = np.array([0, -2.5])
var_110 = np.array([1, 1])

# Profile (1, 1, 1)
sigma_111 = np.array([[0, 1], [1, np.inf], [0, np.inf]])
mu_111 = np.array([3.5, -0.5, -1.5, -3.5])
var_111 = np.array([1, 1, 1, 1])

sigma = [sigma_000, sigma_001, sigma_010, sigma_011,
            sigma_100, sigma_101, sigma_110, sigma_111]

mu = [mu_000, mu_001, mu_010, mu_011,
        mu_100, mu_101, mu_110, mu_111]

var = [var_000, var_001, var_010, var_011,
        var_100, var_101, var_110, var_111]


In [2]:
policies_profiles = {}
policies_profiles_masked = {}
policies_ids_profiles = {}
pi_policies = {}
pi_pools = {}
for k, profile in enumerate(profiles):

    policies_temp = [(i, x) for i, x in enumerate(all_policies) if hasse.policy_to_profile(x) == profile]
    unzipped_temp = list(zip(*policies_temp))
    policies_ids_k = list(unzipped_temp[0])
    policies_k = list(unzipped_temp[1])
    policies_profiles[k] = deepcopy(policies_k)
    policies_ids_profiles[k] = policies_ids_k

    profile_mask = list(map(bool, profile))

    # Mask the empty arms
    for idx, pol in enumerate(policies_k):
        policies_k[idx] = tuple([pol[i] for i in range(M) if profile_mask[i]])
    policies_profiles_masked[k] = policies_k

    if np.sum(profile) > 0:
        pi_pools_k, pi_policies_k = extract_pools.extract_pools(policies_k, sigma[k])
        if len(pi_pools_k.keys()) != mu[k].shape[0]:
            print(f"Profile {k}. Expected {len(pi_pools_k.keys())} pools. Received {mu[k].shape[0]} means.")
        pi_policies[k] = pi_policies_k
        # pi_pools_k has indicies that match with policies_profiles[k]
        # Need to map those indices back to all_policies
        pi_pools[k] = {}
        for x, y in pi_pools_k.items():
            y_full = [policies_profiles[k][i] for i in y]
            y_agg = [all_policies.index(i) for i in y_full]
            pi_pools[k][x] = y_agg
    else:
        pi_policies[k] = {0: 0}
        pi_pools[k] = {0: [0]}

In [3]:
def generate_data(mu, var, n_per_pol, all_policies, pi_policies, M):
    num_data = num_policies * n_per_pol
    X = np.zeros(shape=(num_data, M))
    D = np.zeros(shape=(num_data, 1), dtype='int_')
    y = np.zeros(shape=(num_data, 1))

    idx_ctr = 0
    for k, profile in enumerate(profiles):
        policies_k = policies_profiles[k]

        for idx, policy in enumerate(policies_k):
            policy_idx = [i for i, x in enumerate(all_policies) if x == policy]

            pool_id = pi_policies[k][idx]
            mu_i = mu[k][pool_id]
            var_i = var[k][pool_id]
            y_i = np.random.normal(mu_i, var_i, size=(n_per_pol, 1))

            start_idx = idx_ctr * n_per_pol
            end_idx = (idx_ctr + 1) * n_per_pol

            X[start_idx:end_idx, ] = policy
            D[start_idx:end_idx, ] = policy_idx[0]
            y[start_idx:end_idx, ] = y_i

            idx_ctr += 1

    return X, D, y

In [4]:
num_samples_per_feature = 500

np.random.seed(721)
X, D, y = generate_data(mu, var, num_samples_per_feature, all_policies, pi_policies, M)
policy_means = loss.compute_policy_means(D, y, num_policies)

In [5]:
H = np.inf
theta = 13.3
lamb = 1
R_set, R_profiles = aggregate.RAggregate(M, R, H, D, y, theta, reg=lamb, verbose=True, bruteforce=False)

(0, 0, 0) 12.330368826815754
Profile (0, 0, 0) has 1 objects in Rashomon set
(0, 0, 1) 12.358771154581058
Adaptive
Profile (0, 0, 1) took 0.0016279220581054688 s adaptively
Profile (0, 0, 1) has 2 objects in Rashomon set
(0, 1, 0) 12.358733805947402
Adaptive
Profile (0, 1, 0) took 0.0014541149139404297 s adaptively
Profile (0, 1, 0) has 2 objects in Rashomon set
(0, 1, 1) 12.410226432167857
Adaptive
Profile (0, 1, 1) took 0.004168272018432617 s adaptively
Profile (0, 1, 1) has 4 objects in Rashomon set
(1, 0, 0) 12.389727673487013
Adaptive
Profile (1, 0, 0) took 0.0031599998474121094 s adaptively
Profile (1, 0, 0) has 4 objects in Rashomon set
(1, 0, 1) 12.473944971588377
Adaptive
Profile (1, 0, 1) took 0.012832164764404297 s adaptively
Profile (1, 0, 1) has 8 objects in Rashomon set
(1, 1, 0) 12.472513798995353
Adaptive
Profile (1, 1, 0) took 0.013164997100830078 s adaptively
Profile (1, 1, 0) has 8 objects in Rashomon set
(1, 1, 1) 12.629301267001312
Adaptive
Profile (1, 1, 1) took 0

In [6]:
anchors = AIS.build_anchor_states(R_set, R_profiles, M, R)
prof_idx_of_policy, profiles = AIS.build_profile_index_of_policy(all_policies, hasse.policy_to_profile)

RPS_states = AIS.raggregate_to_states((R_set, R_profiles), profiles)
score_s = AIS.make_score_s_expneg_raw(
    D=D,
    y=y,
    M=M,
    R=R,
    prof_idx_of_policy = prof_idx_of_policy,
    policies=all_policies,
    policy_means=policy_means,
    reg=lamb,
    lattice_edges=None,   # or your edges
    beta=1.0,             # exp(-loss)
    prior_logprob=lambda state: 0.0
)

In [7]:
log_alpha = [score_s(A) for A in anchors]

buckets = AIS.make_p0_buckets_weighted_S0(RPS_states, np.asarray(R,int), log_alpha,
                                      eps1=0.05, eps2=0.25, min_len=1)
log_p0 = lambda z: AIS.log_p0_distance_weighted_S0(z, buckets)

# 3) Run pilot
ladder, ess_ratios = AIS.pilot_adaptive_ladder(
    init_sampler=lambda N: MCMC.init_from_RPS_batch(RPS_states, log_alpha, N, rng_seed=777),
    log_p0=log_p0,
    score_s=score_s,
    N=512,
    ess_target=0.80,
    beta0=0.0, beta1=1.0,
    initial_delta=0.05,
    min_delta=1e-3,
    moves_per_probe=3,   # small mixing at each accepted β (optional)
    min_len=1,
    rng_seed=777
)
print("Adaptive ladder has", len(ladder), "levels; first 10:", ladder[:10])
print("Per-step ESS ratios (len =", len(ess_ratios), "):", ess_ratios[:10])

KeyboardInterrupt: 

In [ ]:
cfg = AIS.AISConfig(n_paths=300, n_levels=20, moves_per_level=5, min_len=1, seed=2)
out_5test = AIS.run_ais_state_streaming(
    anchors=R_set,          # not used by q0, but you may keep for consistency
    score_s=score_s,        # returns exp(-loss(state)) or similar
    cfg=cfg,
    RPS=RPS_states,              # your Rashomon partitions as a list of State
    R_per=R,  # levels per arm (includes control)
    eps1=0.5, eps2=0.75,
    out_jsonl = "AIS_samples_test.jsonl",
    ladder=ladder
)

In [ ]:
import math
import numpy as np
from statistics import NormalDist

_STD_NORMAL = NormalDist()


def normalize_log_weights(logw):
    """
    Stable normalization of log-weights.
    """
    logw = np.asarray(logw, dtype=float)
    m = np.max(logw)
    w = np.exp(logw - m)
    return w / np.sum(w)

import numpy as np

def extract_policy_mu_sigma_nig(
    state,                    # State: list[ProfilePart], length = num_profiles
    D, y,                     # D[:,0] = global policy id; y is (N,) or (N,1)
    policies,                 # global policies list (len P)
    prof_idx_of_policy,       # length-P array: global policy id -> profile k
    R_per,                    # per-arm levels (len M)
    lattice_edges=None,
    mu0=0.0, kappa0=1.0, alpha0=2.0, beta0=2.0,
    seed=None
):
    """
    For a *single* global partition state, extract one (mu_k, sigma_k) for each policy k
    under a Normal-Inverse-Gamma model

    Returns:
      mu   : np.ndarray (K,)
      sigma: np.ndarray (K,)   # std dev samples (sqrt(sigma^2))
    """
    rng = np.random.default_rng(seed)
    P = len(policies)

    # y to 1D
    y1 = y[:, 0] if (isinstance(y, np.ndarray) and y.ndim == 2) else np.asarray(y).ravel()
    pid_all = D[:, 0].astype(int)

    K = int(np.max(prof_idx_of_policy)) + 1

    # Profile -> list of global policy ids (fixed order defines local indices)
    prof_to_global = [[] for _ in range(K)]
    for pid in range(P):
        prof_to_global[int(prof_idx_of_policy[pid])].append(pid)

    prof_policies = [[policies[i] for i in idxs] for idxs in prof_to_global]
    prof_pid_to_local = []
    for k in range(K):
        idxs = prof_to_global[k]
        prof_pid_to_local.append({pid: j for j, pid in enumerate(idxs)})

    # Also split data indices by profile (fast masks)
    prof_data_idx = []
    for k in range(K):
        idxs = np.array(prof_to_global[k], dtype=int)
        if idxs.size == 0:
            prof_data_idx.append(np.array([], dtype=int))
            continue
        mask = np.isin(pid_all, idxs)
        prof_data_idx.append(np.flatnonzero(mask))

    # For each profile, build pools & also pool sufficient stats
    per_prof_poolmap = [None] * K
    per_prof_poolstats = [None] * K
    per_prof_npools = [0] * K

    mu = np.zeros(P, float)
    t_scale = np.zeros(P, float)
    df = np.zeros(P, float)

    for k in range(K):
        idxs = prof_to_global[k]
        if not idxs:
            continue

        sigma_full_k = AIS.assemble_sigma_full_for_profile(state[k], M, np.asarray(R_per, int))
        pi_pools_k, pi_policies_k = extract_pools.extract_pools(prof_policies[k], sigma_full_k, lattice_edges)
        n_pools = len(pi_pools_k)
        per_prof_npools[k] = n_pools
        per_prof_poolmap[k] = pi_policies_k  # local_idx -> pool_id

        # compute sufficient stats by scanning the data rows in this profile
        n = np.zeros(n_pools, int)
        sy = np.zeros(n_pools, float)
        sy2 = np.zeros(n_pools, float)

        didx = prof_data_idx[k]
        pid_k = pid_all[didx]
        y_k = y1[didx]

        pid2loc = prof_pid_to_local[k]
        for pid_obs, y_obs in zip(pid_k, y_k):
            loc = pid2loc[int(pid_obs)]               # local index of this policy in prof_policies[k]
            pool = pi_policies_k[loc]                 # pool id
            n[pool] += 1
            sy[pool] += float(y_obs)
            sy2[pool] += float(y_obs) ** 2

        per_prof_poolstats[k] = (n, sy, sy2)

        idxs = prof_to_global[k]
        if not idxs:
            continue
        pi_policies_k = per_prof_poolmap[k]
        n, sy, sy2 = per_prof_poolstats[k]
        n_pools = per_prof_npools[k]

        # sample a mean for each pool
        pool_mu = np.zeros(n_pools, float)
        pool_t_scale = np.zeros(n_pools, float)
        pool_df = np.zeros(n_pools, float)
        for j in range(n_pools):
            mu_n, k_n, a_n, b_n = MCMC.nig_posterior_params(
                n[j], sy[j], sy2[j],
                mu0=mu0, kappa0=kappa0, alpha0=alpha0, beta0=beta0
            )

            # Marginal posterior:
            # mu | data ~ t_{2a_n}(loc=mu_n, scale=sqrt(b_n / (a_n * k_n)))
            df_n = 2.0 * a_n
            t_scale_n = np.sqrt(b_n / (a_n * k_n))
        
            pool_mu[j] = float(mu_n)
            pool_t_scale[j] = float(np.sqrt(t_scale_n))
            pool_df[j] = float(df_n)

        for local_idx, pid in enumerate(idxs):
            pool_id = pi_policies_k[local_idx]
            mu[pid] = pool_mu[pool_id]
            t_scale[pid] = pool_t_scale[pool_id]
            df[pid] = pool_df[pool_id]

    return mu, t_scale, df


def extract_policy_posteriors_from_ais_sample(ais_sample):
    """
    Parameters
    ----------
    ais_sample : list of dicts
        Each element must have:
          - rec["state"]
          - rec["logw"]

    Returns
    -------
    logw : ndarray, shape (n_particles,)
    mu   : ndarray, shape (n_particles, n_policies)
    sd   : ndarray, shape (n_particles, n_policies)
    """
    logw = []
    mu_list = []
    scale_list = []
    df_list = []

    state_list = ais_sample["terminals"]
    lw_list = ais_sample["logw"]

    for state, lw in zip(state_list, lw_list):
        mu_i, scale_i, df_i = extract_policy_mu_sigma_nig(
            state=state,                    # State: list[ProfilePart], length = num_profiles
            D=D, y=y,                     # D[:,0] = global policy id; y is (N,) or (N,1)
            M=M,
            policies=all_policies,                 # global policies list (len P)
            prof_idx_of_policy=prof_idx_of_policy,       # length-P array: global policy id -> profile k
            R_per=R,                    # per-arm levels (len M)
            lattice_edges=None,
            mu0=0.0, kappa0=1.0, alpha0=2.0, beta0=2.0,
            seed=None
        )

        logw.append(lw)
        mu_list.append(mu_i)
        scale_list.append(scale_i)
        df_list.append(df_i)

    logw = np.asarray(logw, dtype=float)
    mu = np.asarray(mu_list, dtype=float)
    scale = np.asarray(scale_list, dtype=float)
    df = np.asarray(df_list, dtype=float)

    return logw, mu, scale, df


from scipy.stats import t as student_t

def ais_policy_cdf(u, logw, mu_k, scale_k, df_k):
    """
    AIS mixture CDF for one policy/profile k under a mixture of Student-t posteriors:
        F_k(u) = sum_i w_i * T_df_i((u - mu_ik) / scale_ik)
    This is achieved by viewing the F_k(u) as a function of the state x, g(x), then use the 
    theory of important sampling to estimate E[g(x)]

    Parameters
    ----------
    u : float
        Value at which to evaluate the CDF.
    logw : array-like, shape (n_particles,)
        Unnormalized AIS log-weights.
    mu_k : array-like, shape (n_particles,)
        Location parameter for policy/profile k under each AIS particle.
    scale_k : array-like, shape (n_particles,)
        Scale parameter of the Student-t posterior for policy/profile k under each AIS particle.
        This is the t-scale, not the posterior standard deviation.
    df_k : array-like or float
        Degrees of freedom for each AIS particle. Can be scalar if shared.

    Returns
    -------
    float
        Weighted AIS mixture CDF at u.
    """
    w = normalize_log_weights(logw)
    mu_k = np.asarray(mu_k, dtype=float)
    scale_k = np.asarray(scale_k, dtype=float)
    df_k = np.asarray(df_k, dtype=float)

    if df_k.ndim == 0:
        df_k = np.full_like(mu_k, float(df_k), dtype=float)

    vals = np.empty_like(mu_k, dtype=float)

    for i, (m, s, df) in enumerate(zip(mu_k, scale_k, df_k)):
        if s <= 0:
            vals[i] = 1.0 if u >= m else 0.0
        else:
            vals[i] = student_t.cdf((u - m) / s, df=df)

    return float(np.sum(w * vals))

def ais_policy_quantile(logw, mu_k, scale_k, df_k, alpha=0.95, tol=1e-8, maxiter=200):
    """Reverse engineer, using binary dissection to find the quantile value"""
    mu_k = np.asarray(mu_k, dtype=float)
    scale_k = np.asarray(scale_k, dtype=float)
    df_k = np.asarray(df_k, dtype=float)

    if df_k.ndim == 0:
        df_k = np.full_like(mu_k, float(df_k), dtype=float)

    eps = 1e-12
    lo = float(np.min(mu_k - 20.0 * np.maximum(scale_k, eps)))
    hi = float(np.max(mu_k + 20.0 * np.maximum(scale_k, eps)))

    while ais_policy_cdf(lo, logw, mu_k, scale_k, df_k) >= alpha:
        lo -= max(1.0, 0.5 * max(abs(lo), 1.0))

    while ais_policy_cdf(hi, logw, mu_k, scale_k, df_k) < alpha:
        hi += max(1.0, 0.5 * max(abs(hi), 1.0))

    for _ in range(maxiter):
        mid = 0.5 * (lo + hi)
        fmid = ais_policy_cdf(mid, logw, mu_k, scale_k, df_k)
        if fmid < alpha:
            lo = mid
        else:
            hi = mid
        if abs(hi - lo) < tol:
            break

    return 0.5 * (lo + hi)


def ais_quantiles_for_all_policies(ais_sample, alpha=0.95):
    """
    Compute AIS posterior alpha-quantile for every policy.

    Returns
    -------
    dict with:
      - alpha
      - quantiles : shape (n_policies,)
      - logw, mu, sd
    """
    logw, mu, scale, df = extract_policy_posteriors_from_ais_sample(ais_sample=ais_sample)

    n_policies = mu.shape[1]
    q = np.empty(n_policies, dtype=float)

    for k in range(n_policies):
        q[k] = ais_policy_quantile(
            logw=logw,
            mu_k=mu[:, k],
            scale_k=scale[:, k],
            df_k=df[:, k],
            alpha=alpha,
        )

    return {
        "alpha": alpha,
        "quantiles": q,
        "logw": logw,
        "mu": mu,
        "sd": scale,
        "df": df
    }

In [8]:
ais_samples = AIS.load_ais_from_jsonl("AIS_samples_test.jsonl")

In [9]:
ais_samples["logw"]

array([ -7.77771437,  -8.69075688,  -9.85490281,  -9.50202792,
        -9.26589663,  -9.36993005,  -8.88175852,  -8.97102021,
        -9.21493546,  -8.65518956,  -7.71763777,  -8.97753066,
       -10.57028091,  -8.68220171,  -9.88881226,  -6.94065738,
        -7.40705019,  -9.85068651, -10.00694705, -10.13353201,
        -9.27744667,  -9.09613602,  -9.48982169,  -7.92554052,
        -9.51136986,  -9.88572761,  -9.45661   ,  -8.80397595,
        -8.32978999,  -7.96099115,  -8.27332725, -10.66205741,
        -7.84430016, -10.21840945,  -7.72320033, -10.56663258,
        -8.74212464,  -8.63638997,  -8.77493506, -10.06372115,
        -7.67855389,  -9.28142415,  -8.87082042,  -8.8149115 ,
        -8.23321228,  -8.83334143,  -7.1794216 ,  -7.24915916,
        -8.06423992,  -9.90576647,  -8.08689015,  -7.80760395,
       -10.12603506,  -8.73308633,  -7.24807403, -10.35306994,
        -8.6418423 ,  -8.29636783,  -9.93879896,  -8.63004887,
        -8.71318821,  -8.5406377 ,  -7.84814145,  -7.74

In [11]:
out = AIS.ais_quantiles_for_all_policies(
    ais_samples,
    D, y,                     # D[:,0] = global policy id; y is (N,) or (N,1)
    M,
    all_policies,                 # global policies list (len P)
    prof_idx_of_policy,       # length-P array: global policy id -> profile k
    R,                    # per-arm levels (len M)
    lattice_edges=None,
    mu0=0.0, kappa0=1.0, alpha0=2.0, beta0=2.0,
    alpha=0.95,
    seed=None
)

beta_q95 = out["quantiles"]

In [13]:
AIS.ais_quantiles_for_all_policies(
    ais_samples,
    D, y,                     # D[:,0] = global policy id; y is (N,) or (N,1)
    M,
    all_policies,                 # global policies list (len P)
    prof_idx_of_policy,       # length-P array: global policy id -> profile k
    R,                    # per-arm levels (len M)
    lattice_edges=None,
    mu0=0.0, kappa0=1.0, alpha0=2.0, beta0=2.0,
    alpha=0.5,
    seed=None
)["quantiles"]

array([ 0.04162558, -1.98480399, -1.98889732, -2.96388113,  0.36561449,
        0.59165965, -2.95400304,  0.37226166,  0.59888078,  3.55711071,
       -0.81377933, -0.80813646, -1.49846994,  0.9836567 , -0.48540645,
       -1.52150612,  0.98364179, -0.48441872,  3.70662985, -0.68716028,
       -0.75266517, -1.68193156, -1.32018874, -2.33620644, -1.70484075,
       -1.31189782, -2.33419752,  3.76568477, -0.63839115, -0.71198521,
       -1.82726614, -2.22844   , -2.46975322, -1.85269937, -2.22658243,
       -2.46828359])

In [12]:
beta_q95

array([ 0.38748938, -1.67778725, -1.68281401, -2.65946817,  0.78082008,
        2.13956607, -2.64481117,  0.7906054 ,  2.14764519,  3.90372089,
       -0.34259678,  0.0503598 ,  0.12999244,  3.47124543,  1.75528735,
        0.12895372,  3.46976637,  1.75499215,  4.15519132,  0.08874406,
       -0.28445685, -1.14508381,  0.7373803 , -0.39498993, -1.1539145 ,
        0.73807447, -0.39378633,  4.20836874,  1.1906383 , -0.23232462,
       -1.40331536,  0.22372846, -1.02409812, -1.43258298,  0.22520212,
       -1.01400101])